# AICellID — Blood Cell Classifier (Kaggle)

Fine-tunes ResNet18 on the split blood cell dataset.

## One-time Kaggle setup
1. Create a Kaggle account at https://www.kaggle.com.
2. Click **Create → New Dataset**. Upload `pbc_split.zip`. Kaggle auto-extracts the zip. Name it (e.g. `pbc-split`). Set it private if you want.
3. Click **Create → New Notebook**. In the right-hand panel:
   - **+ Add Input → Datasets** → search for and add the dataset you just made.
   - **Settings → Accelerator → GPU T4 x2** (free).
4. **File → Import Notebook** and upload this file.
5. Run all cells. The trained checkpoint appears under **Output** as `cell_classifier.pt` — download it and put it in `backend/models/` in the repo.

## 1. Locate the dataset

In [ ]:
import glob, os
candidates = glob.glob('/kaggle/input/*/')
assert candidates, 'No dataset attached. Use Add Input → Datasets in the right panel.'
DATA_DIR = candidates[0].rstrip('/')
print('using dataset at:', DATA_DIR)
print(os.listdir(DATA_DIR))

## 2. Imports and config

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

BATCH = 64
EPOCHS = 5
LR = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 3. Datasets and dataloaders

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_tf)
val_ds = datasets.ImageFolder(f'{DATA_DIR}/val', transform=val_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2, pin_memory=True)
print('classes:', train_ds.classes)
print('train:', len(train_ds), ' val:', len(val_ds))

## 4. Model

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(train_ds.classes))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

## 5. Train

In [ ]:
best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item()
    acc = correct / len(val_ds)
    print(f'epoch {epoch+1}: train_loss={train_loss:.4f}  val_acc={acc:.4f}')
    best_acc = max(best_acc, acc)
print(f'best val acc: {best_acc:.4f}')

## 6. Save checkpoint

In [ ]:
torch.save({'state_dict': model.state_dict(), 'classes': train_ds.classes}, '/kaggle/working/cell_classifier.pt')
print('saved to /kaggle/working/cell_classifier.pt — download from the Output panel on the right.')